# Imports

In [83]:
from pathlib import Path
import h5py
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.nn as nn
import math

## Globals

In [135]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_FOLDER = Path("data")
NYU_DATASET_FILE = DATA_FOLDER / "nyu_depth_v2_labeled.mat"
KITTI_DATASET_FOLDER = DATA_FOLDER / "kitti"
RESULTS_FOLDER = DATA_FOLDER / "results"

INPUT_SIZE = (224, 304)
BATCH_SIZE = 16 
EPOCH = 40
SEED = 42
LR = 1e-4


## Utils

## Data

In [68]:
class NYUDataset(torch.utils.data.Dataset):
    def __init__(self,mat_file):
        with h5py.File(mat_file,"r") as f:
            self.images = f["images"][:]
            self.depths = f["depths"][:]

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):
        img = self.images[idx]
        depth = self.depths[idx]
        img_t = np.transpose(img,(0,2,1))
        depth_t = depth.T
        image_tensor = torch.from_numpy(img_t).float() / 255
        depth_tensor = torch.from_numpy(depth_t).float()
        image_tensor = F.interpolate(
            image_tensor.unsqueeze(0),
            size = INPUT_SIZE,
            mode = "bilinear",
            align_corners = False
        ).squeeze(0)
        depth_tensor = F.interpolate(
            depth_tensor.unsqueeze(0).unsqueeze(0),
            size = INPUT_SIZE,
            mode = "nearest",
        ).squeeze(0)
        return image_tensor,depth_tensor


        

In [76]:
ds = NYUDataset(NYU_DATASET_FILE)
train_set_size = int(0.8 * len(ds))
val_set_size = len(ds) - train_set_size

#creates training and validation sets
training_set, validation_set = torch.utils.data.random_split(
    ds,
    [train_set_size,val_set_size],
    generator = torch.Generator().manual_seed(SEED)
)

#loads training and validation sets
train_loader = torch.utils.data.DataLoader(training_set , batch_size = BATCH_SIZE, shuffle = True)
val_loader = torch.utils.data.DataLoader(validation_set , batch_size = BATCH_SIZE, shuffle = True)

In [78]:
imgs,dep = next(iter(train_loader))
print(imgs.shape,dep.shape)

torch.Size([16, 3, 224, 304]) torch.Size([16, 1, 224, 304])


## Network

In [110]:
class DepthNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(*list(torchvision.models.resnet18(weights='IMAGENET1K_V1').children())[:-2])
        self.channels = [512, 256, 128, 64, 32, 16]
        self.decoder_blocks = nn.ModuleList([
            nn.Sequential(
            nn.Conv2d(self.channels[i], self.channels[i+1], kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
            )
            for i in range(5)
            ])
        self.final_conv = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        self.sizes = [INPUT_SIZE]
        for _ in range(5):
            h, w = self.sizes[-1]
            self.sizes.append((math.ceil(h / 2), math.ceil(w / 2)))
        self.sizes = self.sizes[::-1]

    def forward(self, x):
        x = self.backbone(x)
        for i in range(5):
            x = F.interpolate(x, size=self.sizes[i+1], mode='bilinear', align_corners=False)
            x = self.decoder_blocks[i](x)
        x = self.final_conv(x)
        return x

In [111]:
model = DepthNet()
print(model.sizes)
print(model(imgs).shape)

[(7, 10), (14, 19), (28, 38), (56, 76), (112, 152), (224, 304)]
torch.Size([16, 1, 224, 304])


## Train

In [121]:
model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(),lr = LR)

def depth_loss(pred, target):
    return torch.mean(torch.abs(pred - torch.log(target)))


In [134]:
for epoch in range(EPOCH):
    model.train()
    train_loss = 0
    for imgs, dep in train_loader:
            imgs, dep = imgs.to(DEVICE), dep.to(DEVICE)
            pred = model(imgs)
            loss = depth_loss(pred, dep)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()


    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, dep in val_loader:
            imgs, dep = imgs.to(DEVICE), dep.to(DEVICE)
            pred = model(imgs)
            val_loss += depth_loss(pred, dep).item()

    print(epoch, train_loss / len(train_loader), val_loss / len(val_loader))

0 0.11960020753210537 0.21574626627721286
1 0.13414438038247906 0.239979088306427
2 0.12428183128980741 0.19918400833481237
3 0.11582401047830712 0.20463135995362935
4 0.12223173463589525 0.20576407642740951
5 0.11669378554167813 0.19803545584804133
6 0.12025852521804914 0.21006070312700773
7 0.11261389516804317 0.19903803734402908
8 0.1063909353050467 0.20288185696852834
9 0.11397222082500588 0.21080756579574786
10 0.11006739684571959 0.2009691407805995
11 0.10144277682451353 0.2043728193170146
12 0.09938807183340805 0.19094470300172506
13 0.10552916390030351 0.20308106745544233
14 0.100458403666542 0.18946814850756996
15 0.09225580810684048 0.19057064150509082
16 0.09286455320168847 0.19159239765844846
17 0.11623289272801517 0.2132013400918559
18 0.10452769931456814 0.2048405110836029
19 0.0890442260528264 0.19356207079009005
20 0.0921731413635489 0.1888227094160883
21 0.09041805493913285 0.20190241462305972
22 0.0925142279226486 0.20321423129031532
23 0.09202787769983893 0.202698561

In [137]:
torch.save(model.state_dict(), RESULTS_FOLDER / "depthnet.pth")

## Evaluation